In [ ]:
pip install scapy pandas scikit-learn joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 17.3 MB/s eta 0:00:00


**1. Feature Extraction**

In [ ]:
from scapy.all import rdpcap, DHCP, BOOTP, Ether
import pandas as pd
from collections import defaultdict

LEGIT_SERVER = "192.168.0.1"

pcaps = [
    "normal_delay.pcap",
    "ettercap_lap2.pcap"
]

msg_map = {
    1: "Discover",
    2: "Offer",
    3: "Request",
    5: "ACK",
    6: "NAK"
}

rows = []

for pcap in pcaps:

    print("\nReading:", pcap)

    packets = rdpcap(pcap)

    xid_data = defaultdict(
        lambda: {
            "discover": 0,
            "offer": 0,
            "request": 0,
            "ack": 0,

            "serverids": set(),
            "servermacs": set(),

            "winner": "Unknown",

            "packet_sizes": [],

            "discover_time": None,
            "offer_time": None,

            "all_times": []
        }
    )

    ################################################

    for pkt in packets:

        if DHCP not in pkt:
            continue

        if BOOTP not in pkt:
            continue

        xid = pkt[BOOTP].xid

        time_now = float(pkt.time)

        options = {}

        try:

            for opt in pkt[DHCP].options:

                if isinstance(opt, tuple):
                    options[opt[0]] = opt[1]

        except:
            continue

        if "message-type" not in options:
            continue

        msg = options["message-type"]

        if isinstance(msg, bytes):
            msg = msg[0]

        msg_name = msg_map.get(
            msg,
            "Unknown"
        )

        serverid = str(
            options.get(
                "server_id",
                "NA"
            )
        )

        srcmac = (
            pkt[Ether].src
            if Ether in pkt
            else "NA"
        )

        ################################################

        xid_data[xid]["packet_sizes"].append(
            len(pkt)
        )

        xid_data[xid]["all_times"].append(
            time_now
        )

        ################################################

        if msg_name == "Discover":

            xid_data[xid]["discover"] += 1

            xid_data[xid][
                "discover_time"
            ] = time_now

        elif msg_name == "Offer":

            xid_data[xid]["offer"] += 1

            xid_data[xid][
                "serverids"
            ].add(serverid)

            xid_data[xid][
                "servermacs"
            ].add(srcmac)

            ##################################
            # FIRST OFFER ONLY
            ##################################

            if xid_data[xid][
                "offer_time"
            ] is None:

                xid_data[xid][
                    "offer_time"
                ] = time_now

        elif msg_name == "Request":

            xid_data[xid]["request"] += 1

        elif msg_name == "ACK":

            xid_data[xid]["ack"] += 1

            xid_data[xid][
                "winner"
            ] = serverid

    ################################################
    # CREATE ONE ROW PER XID
    ################################################

    for xid, data in xid_data.items():

        multiple_offers = 0
        multiple_servers = 0
        multiple_macs = 0

        if data["offer"] > 1:
            multiple_offers = 1

        if len(
            data["serverids"]
        ) > 1:
            multiple_servers = 1

        if len(
            data["servermacs"]
        ) > 1:
            multiple_macs = 1

        ##################################
        # Discover→First Offer delay
        ##################################

        delay = 0

        if (
            data["discover_time"] is not None
            and
            data["offer_time"] is not None
        ):

            delay = (
                data["offer_time"]
                -
                data["discover_time"]
            )

        ##################################

        duration = 0

        if len(
            data["all_times"]
        ) > 1:

            duration = (

                max(
                    data["all_times"]
                )

                -

                min(
                    data["all_times"]
                )
            )

        ##################################

        ratio = 0

        if data["discover"] > 0:

            ratio = (
                data["offer"] +
                data["ack"]
            ) / data["discover"]

        ##################################

        label = 0

        if data["winner"] != LEGIT_SERVER:

            label = 1

        rows.append({

            "xid": xid,

            "discover":
            data["discover"],

            "offer":
            data["offer"],

            "request":
            data["request"],

            "ack":
            data["ack"],

            "server_count":
            len(data["serverids"]),

            "mac_count":
            len(data["servermacs"]),

            "avg_packet_size":
            sum(
                data["packet_sizes"]
            ) / len(
                data["packet_sizes"]
            ),

            "transaction_duration":
            duration,

            "discover_offer_delay":
            delay,

            "multiple_offers_same_xid":
            multiple_offers,

            "multiple_server_reply":
            multiple_servers,

            "multiple_server_macs":
            multiple_macs,

            "discover_offer_ack_ratio":
            ratio,

            "winner":
            data["winner"],

            "label":
            label

        })

df = pd.DataFrame(rows)

df.to_csv(
    "dhcp_ml_features.csv",
    index=False
)

print(df.head())

print("\nSaved CSV")


Reading: normal_delay.pcap


FileNotFoundError: [Errno 2] No such file or directory: 'normal_delay.pcap'

In [ ]:
import pandas as pd
import joblib

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

########################################
# Load CSV
########################################

df = pd.read_csv(
    "dhcp_ml_features.csv"
)

########################################
# Learn threshold from NORMAL traffic
########################################

normal = df[
    df["label"] == 0
]

mean_delay = normal[
    "discover_offer_delay"
].mean()

std_delay = normal[
    "discover_offer_delay"
].std()

threshold = (
    mean_delay
    -
    (2 * std_delay)
)

print(
    "\nDelay Threshold:",
    round(threshold,4)
)

########################################
# Fast Offer Feature
########################################

df["fast_offer"] = (

    df[
        "discover_offer_delay"
    ] < threshold

).astype(int)

########################################
# Encode winner column
########################################

encoder = LabelEncoder()

df["winner"] = (

    encoder.fit_transform(

        df["winner"]

    )

)

########################################
# Features and labels
########################################

X = df.drop(
    "label",
    axis=1
)

y = df["label"]

########################################
# Split
########################################

Xtrain,\
Xtest,\
ytrain,\
ytest = train_test_split(

    X,
    y,

    test_size=0.2,

    random_state=42,

    stratify=y

)

########################################
# Model
########################################

model = RandomForestClassifier(

    n_estimators=200,

    max_depth=8,

    random_state=42

)

model.fit(
    Xtrain,
    ytrain
)

pred = model.predict(
    Xtest
)

########################################
# Save model
########################################

joblib.dump(
    model,
    "dhcp_model.pkl"
)

joblib.dump(
    encoder,
    "winner_encoder.pkl"
)

########################################
# Save predictions
########################################

joblib.dump(
    ytest,
    "ytest.pkl"
)

joblib.dump(
    pred,
    "pred.pkl"
)

print(
"\nModel Saved"
)


Delay Threshold: 0.7934

Model Saved


In [ ]:
import joblib

from sklearn.metrics import(

accuracy_score,

precision_score,

recall_score,

f1_score,

confusion_matrix,

classification_report

)

########################################

ytest=joblib.load(
"ytest.pkl"
)

pred=joblib.load(
"pred.pkl"
)

########################################

print(
"\n=========RESULTS=========\n"
)

print(

"Accuracy:",

round(

accuracy_score(
ytest,
pred
),

4

)

)

print(

"Precision:",

round(

precision_score(
ytest,
pred,
zero_division=0
),

4

)

)

print(

"Recall:",

round(

recall_score(
ytest,
pred,
zero_division=0
),

4

)

)

print(

"F1:",

round(

f1_score(
ytest,
pred,
zero_division=0
),

4

)

)

print(
"\nConfusion Matrix\n"
)

print(

confusion_matrix(
ytest,
pred
)

)

print(
"\nClassification Report\n"
)

print(

classification_report(
ytest,
pred
)

)


=========RESULTS=========

Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1: 1.0

Confusion Matrix

[[34  0]
 [ 0 33]]

Classification Report

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        34
           1       1.00      1.00      1.00        33

    accuracy                           1.00        67
   macro avg       1.00      1.00      1.00        67
weighted avg       1.00      1.00      1.00        67

